# 01 - Exploratory Data Analysis (EDA)
## Indian Liver Patient Dataset (ILPD)

This notebook explores patient demographic distributions, liver function test (LFT) distributions,
missing values, duplicate records, correlations, and class distributions.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_raw, standardise, deduplicate, get_data_summary
from src.config import CANONICAL_COLUMNS, NUM_COLS, REFERENCE_RANGES

### 1. Raw Data Loading & Quality Audit

In [2]:
raw_df = load_raw()
std_df = standardise(raw_df)
print(f"Raw dataset dimensions: {raw_df.shape}")
print("\nMissing values:")
print(std_df.isnull().sum()[std_df.isnull().sum() > 0])

### 2. Deduplication Audit
Exact duplicate patients must be removed prior to train/test splitting to prevent data leakage.

In [3]:
clean_df, n_dups = deduplicate(std_df)
summary = get_data_summary(clean_df)
print(f"Duplicates removed: {n_dups}")
print(f"Target distribution:\n{summary['target_distribution']}")

### 3. Summary Statistics by Clinical Diagnosis
Grouped comparisons illustrate characteristic elevation in bilirubin and transaminases among liver patients.

In [4]:
grouped = clean_df.groupby('target')[NUM_COLS].median().T
grouped.columns = ['Non-Liver Patient (0)', 'Liver Patient (1)']
grouped['Ratio (Liver/Non-Liver)'] = grouped['Liver Patient (1)'] / grouped['Non-Liver Patient (0)']
grouped

### Key EDA Findings:
1. **Class Distribution**: 71.4% liver patients vs. 28.6% non-liver patients.
2. **Right Skew**: Bilirubin, ALP, ALT, and AST exhibit extreme positive skewness, confirming the need for logarithmic (`log1p`) transformations.
3. **Missingness**: Missing values are restricted to 4 records in the `ag_ratio` feature; these are safely imputed by the median inside the scikit-learn pipeline without leakage.